In [62]:
# Code to analyze all elements other than Lead (as it was performed previously) from Maven

In [63]:
import pandas as pd

In [64]:
# Read data
df_maven = pd.read_csv('input/questions_by_model_except_lead.csv',na_filter=False)
len(df_maven)

14074

In [65]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14074 entries, 0 to 14073
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   model_name        14074 non-null  object
 1   question_package  14074 non-null  object
 2   UniqueID          14074 non-null  object
 3   value             14074 non-null  object
 4   count_of_values   14074 non-null  int64 
 5   case_count        14074 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 659.8+ KB


In [66]:
df_maven

,model_name,question_package,UniqueID,value,count_of_values,case_count
0,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_NO,NO,1250,5941891
1,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_UNKNOWN,UNKNOWN,39501,5941891
2,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_YES,YES,24,5941891
3,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_NO,NO,213460,5941891
4,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_UNKNOWN,UNKNOWN,184527,5941891
...,...,...,...,...,...,...
14069,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAP_AROUND_SERVICE_YES,YES,34176,5941891
14070,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAPAROUND_ACCEPTANCE_DATE,12/31/2021,2605,5941891
14071,DiseaseSurveillanceModel_Hep,DEMOGRAPHIC,ZIP,99999,1205519,564938
14072,DiseaseSurveillanceModel_General,DEMOGRAPHIC,ZIP,99999,12956577,5941891


In [67]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:]==answer):
        return question[:start].strip('_')
    else:
        return question

In [68]:
df_maven.loc[:,'parent_question_id']  = [ join_all_except_last(q,a) for q,a in zip(df_maven.loc[:,'UniqueID'],df_maven.loc[:,'value'])]

In [69]:
df_maven

,model_name,question_package,UniqueID,value,count_of_values,case_count,parent_question_id
0,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_NO,NO,1250,5941891,ABX_EKG
1,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_UNKNOWN,UNKNOWN,39501,5941891,ABX_EKG
2,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_YES,YES,24,5941891,ABX_EKG
3,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_NO,NO,213460,5941891,ABXCHEST_YN
4,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_UNKNOWN,UNKNOWN,184527,5941891,ABXCHEST_YN
...,...,...,...,...,...,...,...
14069,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAP_AROUND_SERVICE_YES,YES,34176,5941891,WRAP_AROUND_SERVICE
14070,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAPAROUND_ACCEPTANCE_DATE,12/31/2021,2605,5941891,WRAPAROUND_ACCEPTANCE_DATE
14071,DiseaseSurveillanceModel_Hep,DEMOGRAPHIC,ZIP,99999,1205519,564938,ZIP
14072,DiseaseSurveillanceModel_General,DEMOGRAPHIC,ZIP,99999,12956577,5941891,ZIP


In [70]:
df_maven.loc[:,'choice_value'] = [v if u!=p else '' for v,u,p in zip(df_maven.loc[:,'value'],df_maven.loc[:,'UniqueID'],df_maven.loc[:,'parent_question_id'])]

In [71]:
df_maven

,model_name,question_package,UniqueID,value,count_of_values,case_count,parent_question_id,choice_value
0,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_NO,NO,1250,5941891,ABX_EKG,NO
1,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_UNKNOWN,UNKNOWN,39501,5941891,ABX_EKG,UNKNOWN
2,DiseaseSurveillanceModel_General,CLINICAL,ABX_EKG_YES,YES,24,5941891,ABX_EKG,YES
3,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_NO,NO,213460,5941891,ABXCHEST_YN,NO
4,DiseaseSurveillanceModel_General,CLINICAL,ABXCHEST_YN_UNKNOWN,UNKNOWN,184527,5941891,ABXCHEST_YN,UNKNOWN
...,...,...,...,...,...,...,...,...
14069,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAP_AROUND_SERVICE_YES,YES,34176,5941891,WRAP_AROUND_SERVICE,YES
14070,DiseaseSurveillanceModel_General,CONTROL_ISO_QUAR,WRAPAROUND_ACCEPTANCE_DATE,12/31/2021,2605,5941891,WRAPAROUND_ACCEPTANCE_DATE,
14071,DiseaseSurveillanceModel_Hep,DEMOGRAPHIC,ZIP,99999,1205519,564938,ZIP,
14072,DiseaseSurveillanceModel_General,DEMOGRAPHIC,ZIP,99999,12956577,5941891,ZIP,


In [72]:
# Group by parent question to get completion rates
df_group = df_maven.groupby(by= ['model_name','question_package','parent_question_id','case_count'])['count_of_values'].sum().reset_index(name= 'count_of_values')

In [73]:
df_group

,model_name,question_package,parent_question_id,case_count,count_of_values
0,ClusterModel,ADMINISTRATIVE,AFTER_OUTBREAK_RECOMMENDATIONS,4541,10
1,ClusterModel,ADMINISTRATIVE,ALL_ECOLI,4541,240
2,ClusterModel,ADMINISTRATIVE,ALL_ENT,4541,15528
3,ClusterModel,ADMINISTRATIVE,ALL_FLU,4541,6780
4,ClusterModel,ADMINISTRATIVE,ALL_GEN,4541,444
...,...,...,...,...,...
10472,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,STD_SCREENING_PARTNER,361,242
10473,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,STD_SCREENING_PARTNER_ID,361,3
10474,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER,361,3882
10475,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_STD,361,2904


In [74]:
df_group.loc[:,'fill_rate'] = [round(n*100.0/d,3) for n,d in zip(df_group.loc[:,'count_of_values'],df_group.loc[:,'case_count'])]

In [75]:
df_group

,model_name,question_package,parent_question_id,case_count,count_of_values,fill_rate
0,ClusterModel,ADMINISTRATIVE,AFTER_OUTBREAK_RECOMMENDATIONS,4541,10,0.220
1,ClusterModel,ADMINISTRATIVE,ALL_ECOLI,4541,240,5.285
2,ClusterModel,ADMINISTRATIVE,ALL_ENT,4541,15528,341.951
3,ClusterModel,ADMINISTRATIVE,ALL_FLU,4541,6780,149.306
4,ClusterModel,ADMINISTRATIVE,ALL_GEN,4541,444,9.778
...,...,...,...,...,...,...
10472,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,STD_SCREENING_PARTNER,361,242,67.036
10473,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,STD_SCREENING_PARTNER_ID,361,3,0.831
10474,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER,361,3882,1075.346
10475,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_STD,361,2904,804.432


In [76]:
#df_group = df_group[['model_name','question_package','parent_question_id','fill_rate']]

In [77]:
df_group.to_csv('output/all_elements_analysis_except_lead.csv',index=False)

In [78]:
df_std = df_group[df_group['model_name']=='DiseaseSurveillanceModel_STD']

In [79]:
df_general = df_group[df_group['model_name']=='DiseaseSurveillanceModel_General']
len(df_general)

7477

In [80]:
df_hep = df_group[df_group['model_name']=='DiseaseSurveillanceModel_Hep']
len(df_hep)

1388

In [81]:
df_cluster = df_group[df_group['model_name']=='ClusterModel']
len(df_cluster)

207

In [82]:
df_portal = df_group[df_group['model_name']=='PortalApplicationModel']
len(df_portal)

41

In [83]:
df_std.to_csv('output/std_element_analysis.csv',index=False)
df_general.to_csv('output/general_element_analysis.csv',index=False)
df_hep.to_csv('output/hepatitis_element_analysis.csv',index=False)
df_cluster.to_csv('output/cluster_element_analysis.csv',index=False)
df_portal.to_csv('output/portal_element_analysis.csv',index=False)

In [84]:
#STOP

In [45]:
# Merge with file from Jennifer to get the display text column

In [46]:
df_other = pd.read_csv('input/LEAD_Migrated_fields_Maven_DB_JL.csv',na_filter=False)
len(df_other)

1473

In [47]:
df_other

,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText,Fill Rate,ELR Fields
0,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_ADULT,AGE_FLAG,ADULT,Adult Labs,,
1,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_CHILD,AGE_FLAG,CHILD,Child Labs,,
2,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_MIXED,AGE_FLAG,MIXED,Child and Adult Mixed Labs,,
3,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_UNKNOWN,AGE_FLAG,UNKNOWN,Missing Information,,
4,Childhood_Lead_Child_Model,ADMINISTRATIVE,ANY_OVER_5_OUTPUT,,,This Is a Case (>=5 Capillary or Venous):,,
...,...,...,...,...,...,...,...,...
1468,Childhood_Lead_Investigation_Model,REMEDIATION_ABATEMENT,REMEDIATION_YES,REMEDIATION,YES,Yes,,
1469,Childhood_Lead_Investigation_Model,REMEDIATION_ABATEMENT,SIGNED_AGREEMENT_NO,SIGNED_AGREEMENT,NO,No,,
1470,Childhood_Lead_Investigation_Model,REMEDIATION_ABATEMENT,SIGNED_AGREEMENT_YES,SIGNED_AGREEMENT,YES,Yes,,
1471,Childhood_Lead_Investigation_Model,REMEDIATION_ABATEMENT,VACANT_NO,VACANT,NO,No,,


In [48]:
# Merger df_group with df_other on parent_question_id
df_merge = pd.merge(df_group,df_other,)

MergeError: No common columns to perform merge on. Merge options: left_on=None, right_on=None, left_index=False, right_index=False